# 12_model_baseline_comparison_rebuild_260514

Full rebuild of Step 12 with fixed-parameter model-family comparison plus top-k operating diagnostics and decile calibration summaries.

In [1]:
from pathlib import Path
from datetime import datetime
import subprocess, warnings, zipfile, json, importlib, math
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

STEP = '12_model_baseline_comparison_rebuild_260514'
EXPECTED_ROOTS = {'C:/Code/ott-churn-prediction', 'C:\\Code\\ott-churn-prediction'}
actual_root = subprocess.check_output(['git', 'rev-parse', '--show-toplevel'], text=True).strip()
print('repo root:', actual_root)
if actual_root not in EXPECTED_ROOTS:
    raise SystemExit(f'STOP: repo root mismatch: {actual_root}')

ROOT = Path(actual_root)
PARK = ROOT / 'park.ingyeom'
NOTEBOOK = PARK / 'notebook' / STEP / f'{STEP}.ipynb'
NOTE = PARK / 'note.md'
BASE_MODEL = PARK / 'reports' / 'models' / STEP
BASE_FIG = PARK / 'reports' / 'figures' / STEP
ZIP_PATH = PARK / 'zip' / f'{STEP}_review_package.zip'

def inside_park(path):
    try:
        Path(path).resolve().relative_to(PARK.resolve())
        return True
    except Exception:
        return False

for p in [NOTEBOOK, NOTE, BASE_MODEL, BASE_FIG, ZIP_PATH]:
    assert inside_park(p), f'outside park.ingyeom blocked: {p}'

def choose_dir(base):
    base.mkdir(parents=True, exist_ok=True)
    if any(base.iterdir()):
        d = base / f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        d.mkdir(parents=True, exist_ok=False)
        return d
    return base

MODEL_DIR = choose_dir(BASE_MODEL)
FIG_DIR = choose_dir(BASE_FIG)
print('actual model output folder:', MODEL_DIR)
print('actual figure output folder:', FIG_DIR)

def all_pass(path):
    if not Path(path).exists(): return False
    x = pd.read_csv(path)
    return 'status' in x.columns and x['status'].fillna('').eq('PASS').all()

def detect_10():
    req = ['10_final_checks.csv','10_feature_eda_catalog.csv','10_focus_feature_deep_dive_summary.csv','10_handoff_to_11_and_17.csv','10_open_risks_for_next_steps.csv']
    base = PARK/'reports/eda/10_feature_eda_260513'
    if all((base/n).exists() for n in req) and all_pass(base/'10_final_checks.csv'): return base
    runs = [d for d in base.glob('run_*') if d.is_dir() and all((d/n).exists() for n in req) and all_pass(d/'10_final_checks.csv')] if base.exists() else []
    return sorted(runs, key=lambda p:p.name)[-1] if runs else None

def detect_11b():
    req = ['11b_final_checks.csv','11b_modeling_input_contract.csv','11b_feature_ladder_definition.csv','11b_ladder_contamination_check.csv','11b_dataset_scope_definition.csv','11b_model_registry.csv','11b_cv_summary_metrics.csv','11b_best_baseline_by_scope.csv','11b_ladder_growth_summary.csv','11b_train_valid_gap_audit.csv','11b_score_orientation_policy.csv','11b_open_risks_for_next_steps.csv','11b_handoff_to_12_model_comparison.csv']
    base = PARK/'reports/models/11b_baseline_growth_history_ladder_fix_260514'
    c = []
    if base.exists() and all((base/n).exists() for n in req): c.append(base)
    if base.exists(): c += [d for d in base.glob('run_*') if d.is_dir() and all((d/n).exists() for n in req)]
    v = [d for d in c if all_pass(d/'11b_final_checks.csv')]
    return sorted(v, key=lambda p:p.name)[-1] if v else None

def detect_semantic():
    req = ['11b_semantic_final_checks.csv','11b_canonical_status_decision.csv','11b_ladder_semantic_classification.csv','11b_ladder_interpretation_guardrail.csv','11b_handoff_to_12_semantic_requirements.csv']
    base = PARK/'reports/audits/11b_semantic_validation_and_interpretation_patch_260514'
    c = []
    if base.exists() and all((base/n).exists() for n in req): c.append(base)
    if base.exists(): c += [d for d in base.glob('run_*') if d.is_dir() and all((d/n).exists() for n in req)]
    v = [d for d in c if all_pass(d/'11b_semantic_final_checks.csv')]
    return sorted(v, key=lambda p:p.name)[-1] if v else None

P = {
 'primary': PARK/'reports/audits/06_common_preprocessing_and_final_cohort_260513/06_primary_main_cohort_conservative_features.csv',
 'index': PARK/'reports/audits/06_common_preprocessing_and_final_cohort_260513/06_primary_main_cohort_index.csv',
 'canon': PARK/'reports/audits/05b_column_role_dictionary_patch_260513/05b_canonical_column_role_dictionary.csv',
 'safe': PARK/'reports/audits/05b_column_role_dictionary_patch_260513/05b_conservative_safe_candidate_columns.csv',
 'review': PARK/'reports/audits/05b_column_role_dictionary_patch_260513/05b_review_required_columns.csv',
 'forbid': PARK/'reports/audits/05b_column_role_dictionary_patch_260513/05b_forbidden_drop_columns.csv',
 'aarr': PARK/'reports/audits/07_AARRR_feature_mapping_260513/07_AARRR_mapping_conservative_features.csv'}
P09B = PARK/'reports/audits/09b_raw_view_window_validation_260514/run_20260514_130402'
REQ09B = ['09b_final_checks.csv','09b_core_usage_recalculation_comparison.csv','09b_window_validation_decision.csv']
P10, P11B, PSEM = detect_10(), detect_11b(), detect_semantic()
OLD12 = PARK/'reports/models/12_model_baseline_comparison_260513'
required = list(P.values()) + [P09B/n for n in REQ09B]
if P10: required += [P10/n for n in ['10_final_checks.csv','10_feature_eda_catalog.csv','10_focus_feature_deep_dive_summary.csv','10_handoff_to_11_and_17.csv','10_open_risks_for_next_steps.csv']]
if P11B: required += [P11B/n for n in ['11b_final_checks.csv','11b_modeling_input_contract.csv','11b_feature_ladder_definition.csv','11b_ladder_contamination_check.csv','11b_dataset_scope_definition.csv','11b_model_registry.csv','11b_cv_summary_metrics.csv','11b_best_baseline_by_scope.csv','11b_ladder_growth_summary.csv','11b_train_valid_gap_audit.csv','11b_score_orientation_policy.csv','11b_open_risks_for_next_steps.csv','11b_handoff_to_12_model_comparison.csv']]
if PSEM: required += [PSEM/n for n in ['11b_semantic_final_checks.csv','11b_canonical_status_decision.csv','11b_ladder_semantic_classification.csv','11b_ladder_interpretation_guardrail.csv','11b_handoff_to_12_semantic_requirements.csv']]
missing = [str(p) for p in required if not p.exists()]
pre=[]
def add_pre(n, ok, val='', note=''):
    pre.append({'check_name':n,'status':'PASS' if ok else 'FAIL','value':val,'note':note,'actual_repo_root':actual_root,'detected_09b_output_folder':str(P09B),'detected_10_output_folder':str(P10 or ''),'detected_11b_model_output_folder':str(P11B or ''),'detected_11b_semantic_patch_folder':str(PSEM or ''),'actual_model_output_folder':str(MODEL_DIR),'actual_figure_output_folder':str(FIG_DIR)})
for n,ok,val in [('repo_root_checked',True,actual_root),('repo_root_matches_expected',actual_root in EXPECTED_ROOTS,actual_root),('all_required_input_files_exist',not missing,';'.join(missing[:20])),('detected_09b_output_folder',P09B.exists(),P09B),('detected_10_output_folder',P10 is not None,P10),('detected_11b_model_output_folder',P11B is not None,P11B),('detected_11b_semantic_patch_folder',PSEM is not None,PSEM),('09b_final_checks_all_PASS',all_pass(P09B/'09b_final_checks.csv'),''),('10_final_checks_all_PASS',P10 is not None and all_pass(P10/'10_final_checks.csv'),''),('11b_final_checks_all_PASS',P11B is not None and all_pass(P11B/'11b_final_checks.csv'),''),('11b_semantic_final_checks_all_PASS',PSEM is not None and all_pass(PSEM/'11b_semantic_final_checks.csv'),''),('output_folder_inside_park_ingyeom',inside_park(MODEL_DIR),MODEL_DIR),('figure_folder_inside_park_ingyeom',inside_park(FIG_DIR),FIG_DIR)]: add_pre(n,ok,str(val))
can = not missing and P10 is not None and P11B is not None and PSEM is not None and all_pass(P09B/'09b_final_checks.csv') and all_pass(P10/'10_final_checks.csv') and all_pass(P11B/'11b_final_checks.csv') and all_pass(PSEM/'11b_semantic_final_checks.csv')
add_pre('can_proceed',can,str(can))
pd.DataFrame(pre).to_csv(MODEL_DIR/'12r_preflight_input_validation.csv',index=False,encoding='utf-8-sig')
if not can:
    (MODEL_DIR/'README.md').write_text('# Step 12r preflight failed\n\nSee `12r_preflight_input_validation.csv`.\n',encoding='utf-8')
    raise SystemExit('STOP: Step 12r preflight failed')

df = pd.read_csv(P['primary']); safe = pd.read_csv(P['safe']); review = pd.read_csv(P['review']); forbid = pd.read_csv(P['forbid']); aarr = pd.read_csv(P['aarr'])
ladder11b = pd.read_csv(P11B/'11b_feature_ladder_definition.csv'); best11b = pd.read_csv(P11B/'11b_best_baseline_by_scope.csv'); sem_dec = pd.read_csv(PSEM/'11b_canonical_status_decision.csv')
TARGET,SPLIT,GROUP='is_repurchase','is_promotion','USER_KEY'; BLOCK={GROUP,'source_row_number',TARGET,'repurchase_score','churn_risk'}
review_cols=set(review['column_name'].dropna().astype(str)); forbid_cols=set(forbid['column_name'].dropna().astype(str))
safe_features=[c for c in safe['column_name'].dropna().astype(str) if c in df.columns and c not in BLOCK and c!=SPLIT]
meta={str(r['column_name']):{'feature_family':r.get('feature_family',''),'AARRR_stage':r.get('AARRR_stage_primary','')} for _,r in aarr.iterrows()}
def feats_for(scope,step):
    r=ladder11b[(ladder11b.dataset_scope.eq(scope))&(ladder11b.ladder_step.eq(step))]
    return [] if r.empty else [x for x in str(r.iloc[0].feature_names).split(';') if x]
scope_features={'overall_without_promotion':feats_for('overall_without_promotion','L4_all_conservative_behavior'),'overall_with_promotion':feats_for('overall_with_promotion','L5_all_conservative_plus_promotion_indicator'),'promotion_only':feats_for('promotion_only','L4_all_conservative_behavior'),'nonpromotion_only':feats_for('nonpromotion_only','L4_all_conservative_behavior')}
scopes={'overall_without_promotion':df.index.to_numpy(),'overall_with_promotion':df.index.to_numpy(),'promotion_only':df.index[df[SPLIT]==1].to_numpy(),'nonpromotion_only':df.index[df[SPLIT]==0].to_numpy()}
warns=[]
def warn(tp,scope='',model='',fold='',severity='WARNING',msg=''):
    warns.append({'warning_type':tp,'dataset_scope':scope,'model_name':model,'fold':fold,'severity':severity,'message':msg,'actual_model_output_folder':str(MODEL_DIR),'actual_figure_output_folder':str(FIG_DIR)})

pd.DataFrame([{'old_step_11_status':'deprecated','old_step_11_reason':'ladder contamination: diff_between_w3_w2 was included in L2','old_step_12_path':str(OLD12),'old_step_12_status':'pre-rebuild/deprecated' if OLD12.exists() else 'not found','old_step_12_reason':'AUC-centered comparison lacked required operating top-k and calibration diagnostics','11b_used_as_canonical':'yes','11b_semantic_patch_applied':'yes','old_step_11_metrics_used':'no','old_step_12_metrics_used':'no, exclusion audit only','current_12r_status_after_validation':'canonical Step 12 if final checks pass'}]).to_csv(MODEL_DIR/'12r_old_11_and_old_12_exclusion_audit.csv',index=False,encoding='utf-8-sig')
scope_rows=[]
for scope,idx in scopes.items():
    d=df.loc[idx]; n=len(d); pos=int((d[TARGET]==1).sum())
    scope_rows.append({'dataset_scope':scope,'row_count':n,'target_distribution':json.dumps(d[TARGET].value_counts().to_dict(),ensure_ascii=False),'repurchase_rate':pos/n,'unique_USER_KEY_count':d[GROUP].nunique(),'duplicated_USER_KEY_extra_rows':n-d[GROUP].nunique(),'feature_set_used':'L5 all conservative + is_promotion' if scope=='overall_with_promotion' else 'L4 all conservative behavior','feature_count':len(scope_features[scope]),'is_promotion_feature_used':scope=='overall_with_promotion','reason':'12r compares model families on canonical conservative feature set','caution':'row-level subscription-event unit; no final threshold or segmentation'})
pd.DataFrame(scope_rows).to_csv(MODEL_DIR/'12r_dataset_scope_definition.csv',index=False,encoding='utf-8-sig')
pd.DataFrame([{'input_table':str(P['primary']),'row_count':len(df),'conservative_feature_count':len(safe_features),'target_distribution':json.dumps(df[TARGET].value_counts().to_dict(),ensure_ascii=False),'promotion_distribution':json.dumps(df[SPLIT].value_counts().to_dict(),ensure_ascii=False),'dataset_scope_counts':json.dumps({r['dataset_scope']:r['row_count'] for r in scope_rows},ensure_ascii=False),'review_columns_excluded':True,'forbidden_columns_excluded':True,'group_key':GROUP,'score_orientation':'repurchase_score=P(is_repurchase=1); churn_risk=1-repurchase_score','operating_event':'non-repurchase = is_repurchase=0','topk_ranking':'churn_risk descending','09b_window_validation_status':'PASS','11b_canonical_status':sem_dec.loc[0,'canonical_after_patch'],'semantic_guardrail_status':'PASS'}]).to_csv(MODEL_DIR/'12r_modeling_input_contract.csv',index=False,encoding='utf-8-sig')
fs=[]
for scope,feats in scope_features.items():
    for f in feats:
        fs.append({'dataset_scope':scope,'feature_name':f,'included':'yes','reason':'11b canonical L5 feature' if f==SPLIT else '11b canonical conservative feature','feature_family':'split_indicator' if f==SPLIT else meta.get(f,{}).get('feature_family',''),'AARRR_stage':'comparison_split' if f==SPLIT else meta.get(f,{}).get('AARRR_stage',''),'from_conservative_safe':'no' if f==SPLIT else ('yes' if f in safe_features else 'no'),'is_review_column':'yes' if f in review_cols else 'no','is_forbidden_column':'yes' if f in forbid_cols and f!=SPLIT else 'no','caution':'is_promotion only allowed in overall_with_promotion' if f==SPLIT else 'standard conservative feature'})
pd.DataFrame(fs).to_csv(MODEL_DIR/'12r_feature_set_by_scope.csv',index=False,encoding='utf-8-sig')

def make_models():
    rows=[]; models={}
    specs=[('LogisticRegression','sklearn','required',Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler()),('model',LogisticRegression(max_iter=2000,solver='lbfgs'))]),'max_iter=2000, solver=lbfgs'),('HistGradientBoosting','sklearn','required',Pipeline([('imputer',SimpleImputer(strategy='median')),('model',HistGradientBoostingClassifier(random_state=42))]),'random_state=42'),('RandomForest','sklearn','required',Pipeline([('imputer',SimpleImputer(strategy='median')),('model',RandomForestClassifier(n_estimators=300,max_depth=6,min_samples_leaf=20,n_jobs=-1,random_state=42))]),'n_estimators=300,max_depth=6,min_samples_leaf=20,n_jobs=-1,random_state=42'),('GradientBoosting','sklearn','required',Pipeline([('imputer',SimpleImputer(strategy='median')),('model',GradientBoostingClassifier(random_state=42))]),'random_state=42'),('ExtraTrees','sklearn','required',Pipeline([('imputer',SimpleImputer(strategy='median')),('model',ExtraTreesClassifier(n_estimators=300,max_depth=6,min_samples_leaf=20,n_jobs=-1,random_state=42))]),'n_estimators=300,max_depth=6,min_samples_leaf=20,n_jobs=-1,random_state=42')]
    for name,pkg,req,model,params in specs:
        models[name]=model; rows.append({'model_name':name,'package':pkg,'import_available':'yes','required_or_optional':req,'will_run':'yes','unavailable_reason':'','fixed_parameters':params,'tuning_performed':'no'})
    for name,modname,clsname in [('LightGBM','lightgbm','LGBMClassifier'),('XGBoost','xgboost','XGBClassifier'),('CatBoost','catboost','CatBoostClassifier')]:
        try:
            mod=importlib.import_module(modname); cls=getattr(mod,clsname)
            if name=='LightGBM': model=Pipeline([('imputer',SimpleImputer(strategy='median')),('model',cls(n_estimators=300,learning_rate=0.05,num_leaves=31,subsample=0.9,colsample_bytree=0.9,random_state=42,n_jobs=-1,verbosity=-1))]); params='n_estimators=300,learning_rate=0.05,num_leaves=31,subsample=0.9,colsample_bytree=0.9,random_state=42,n_jobs=-1'
            elif name=='XGBoost': model=Pipeline([('imputer',SimpleImputer(strategy='median')),('model',cls(n_estimators=300,learning_rate=0.05,max_depth=4,subsample=0.9,colsample_bytree=0.9,eval_metric='logloss',random_state=42,n_jobs=-1))]); params='n_estimators=300,learning_rate=0.05,max_depth=4,subsample=0.9,colsample_bytree=0.9,eval_metric=logloss,random_state=42,n_jobs=-1'
            else: model=Pipeline([('imputer',SimpleImputer(strategy='median')),('model',cls(iterations=300,learning_rate=0.05,depth=4,random_seed=42,verbose=False))]); params='iterations=300,learning_rate=0.05,depth=4,random_seed=42,verbose=False'
            models[name]=model; rows.append({'model_name':name,'package':modname,'import_available':'yes','required_or_optional':'optional','will_run':'yes','unavailable_reason':'','fixed_parameters':params,'tuning_performed':'no'})
        except Exception as e:
            warn('optional_model_unavailable',model=name,severity='INFO',msg=str(e)); rows.append({'model_name':name,'package':modname,'import_available':'no','required_or_optional':'optional','will_run':'no','unavailable_reason':str(e),'fixed_parameters':'fixed optional config if installed','tuning_performed':'no'})
    return models,pd.DataFrame(rows)
models,avail=make_models(); avail.to_csv(MODEL_DIR/'12r_model_availability.csv',index=False,encoding='utf-8-sig')

cv_rows=[]; metric_rows=[]; oof={}; splits={}
for scope,idx in scopes.items():
    d=df.loc[idx].reset_index(drop=False).rename(columns={'index':'orig_index'}); y=d[TARGET].astype(int).to_numpy(); g=d[GROUP].to_numpy()
    try: sp=list(StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=42).split(np.zeros(len(d)),y,groups=g))
    except Exception as e: warn('cv_scope_failed',scope,severity='FAIL',msg=str(e)); splits[scope]=None; continue
    ok=[]
    for fold,(tr,va) in enumerate(sp,1):
        ov=len(set(g[tr]).intersection(set(g[va]))); both=len(np.unique(y[va]))==2; status='PASS' if ov==0 and both else 'FAIL'
        if status!='PASS': warn('cv_fold_failed',scope,fold=fold,severity='FAIL',msg=f'overlap={ov};both_classes={both}')
        cv_rows.append({'dataset_scope':scope,'fold':fold,'train_rows':len(tr),'valid_rows':len(va),'train_repurchase_rate':y[tr].mean(),'valid_repurchase_rate':y[va].mean(),'train_unique_USER_KEY':len(set(g[tr])),'valid_unique_USER_KEY':len(set(g[va])),'group_overlap_count':ov,'valid_class_both_classes':both,'promotion_rate_if_applicable':d.loc[va,SPLIT].mean(),'status':status})
        if status=='PASS': ok.append((tr,va))
    splits[scope]=(d,ok) if len(ok)==5 else None
pd.DataFrame(cv_rows).to_csv(MODEL_DIR/'12r_cv_split_audit.csv',index=False,encoding='utf-8-sig')
for scope,pack in splits.items():
    if pack is None: continue
    d,sp=pack; feats=[]; bad=[]
    for f in scope_features[scope]:
        if f in BLOCK or (f==SPLIT and scope!='overall_with_promotion'): bad.append(f); continue
        x=pd.to_numeric(d[f],errors='coerce')
        if d[f].notna().sum() and x.notna().sum()==0: bad.append(f)
        else: feats.append(f)
    if bad: warn('non_numeric_or_blocked_feature_excluded',scope,msg=';'.join(bad))
    X=d[feats].apply(pd.to_numeric,errors='coerce'); y=d[TARGET].astype(int).to_numpy(); orig=d.orig_index.to_numpy()
    for model_name,model in models.items():
        pred=np.full(len(d),np.nan); fold_id=np.full(len(d),np.nan); trains=[]; vals=[]; aps=[]; brs=[]
        for fold,(tr,va) in enumerate(sp,1):
            mdl=clone(model)
            try:
                with warnings.catch_warnings(record=True) as caught:
                    warnings.simplefilter('always'); mdl.fit(X.iloc[tr],y[tr]); p_tr=mdl.predict_proba(X.iloc[tr])[:,1]; p_va=mdl.predict_proba(X.iloc[va])[:,1]
                for w in caught: warn('model_fit_warning',scope,model_name,fold,msg=str(w.message))
                ta=roc_auc_score(y[tr],p_tr); vaa=roc_auc_score(y[va],p_va); ap=average_precision_score(y[va],p_va); br=brier_score_loss(y[va],p_va)
                pred[va]=p_va; fold_id[va]=fold; trains.append(ta); vals.append(vaa); aps.append(ap); brs.append(br)
                metric_rows.append({'dataset_scope':scope,'model_name':model_name,'fold':fold,'train_auc':ta,'valid_auc':vaa,'train_valid_gap':ta-vaa,'valid_average_precision':ap,'valid_brier_score':br,'train_row_count':len(tr),'valid_row_count':len(va),'feature_count':len(feats),'warning':'; '.join(str(w.message) for w in caught)})
            except Exception as e: warn('model_fit_failed',scope,model_name,fold,severity='FAIL',msg=str(e))
        if np.isfinite(pred).all(): oof[(scope,model_name)]={'orig_index':orig,'fold':fold_id,'pred':pred,'y':y,'feature_count':len(feats),'trains':trains,'vals':vals,'aps':aps,'brs':brs}
pd.DataFrame(metric_rows).to_csv(MODEL_DIR/'12r_model_comparison_fold_metrics.csv',index=False,encoding='utf-8-sig')

def operating_metrics(y,p):
    churn=1-p; non=(y==0).astype(int); n=len(y); base=non.mean(); total=non.sum(); out={}
    for pct,label in [(0.10,'top10'),(0.20,'top20')]:
        k=max(1,int(math.ceil(n*pct))); idx=np.argsort(-churn)[:k]; cnt=int(non[idx].sum()); prec=cnt/k; rec=cnt/total if total else np.nan; lift=prec/base if base else np.nan
        out[f'precision@{label}%_churn_risk']=prec; out[f'recall@{label}%_churn_risk']=rec; out[f'lift@{label}%']=lift; out[f'{label}_row_count']=k; out[f'{label}_nonrepurchase_count']=cnt; out[f'observed_nonrepurchase_rate_in_{label}']=prec
    out['baseline_nonrepurchase_rate']=base; return out
summary=[]; op_rows=[]; dec_rows=[]
for (scope,model_name),rec in oof.items():
    tr=np.array(rec['trains']); va=np.array(rec['vals']); gap=tr-va; p=rec['pred']; y=rec['y']; churn=1-p
    op=operating_metrics(y,p); op_rows.append({'dataset_scope':scope,'model_name':model_name,**op})
    tmp=pd.DataFrame({'y':y,'repurchase_score':p,'churn_risk':churn})
    tmp['repurchase_decile']=pd.qcut(tmp['repurchase_score'].rank(method='first',ascending=False),10,labels=False)+1
    tmp['risk_decile']=pd.qcut(tmp['churn_risk'].rank(method='first',ascending=False),10,labels=False)+1
    for dec,g in tmp.groupby('repurchase_decile'):
        dec_rows.append({'dataset_scope':scope,'model_name':model_name,'score_type':'repurchase_score','decile':int(dec),'row_count':len(g),'mean_score':g.repurchase_score.mean(),'observed_repurchase_rate':g.y.mean(),'observed_nonrepurchase_rate':1-g.y.mean(),'mean_churn_risk':np.nan})
    for dec,g in tmp.groupby('risk_decile'):
        dec_rows.append({'dataset_scope':scope,'model_name':model_name,'score_type':'churn_risk','decile':int(dec),'row_count':len(g),'mean_score':g.churn_risk.mean(),'observed_repurchase_rate':g.y.mean(),'observed_nonrepurchase_rate':1-g.y.mean(),'mean_churn_risk':g.churn_risk.mean()})
    mg=float(np.nanmean(gap)); sd=float(np.nanstd(va,ddof=1)) if len(va)>1 else 0.0; oauc=roc_auc_score(y,p); oap=average_precision_score(y,p); obr=brier_score_loss(y,p)
    if mg>=0.12: status='overfit_warning'; caution='Large train-valid gap; not final.'; warn('overfit_warning',scope,model_name,msg=caution)
    elif sd>=0.06: status='unstable'; caution='High fold variability.'; warn('instability_warning',scope,model_name,msg=caution)
    elif oauc<0.55: status='weak'; caution='Weak baseline signal.'
    else: status='usable_candidate'; caution='Fixed-parameter candidate only.'
    summary.append({'dataset_scope':scope,'model_name':model_name,'feature_count':rec['feature_count'],'n_folds_completed':len(va),'mean_valid_auc':float(np.nanmean(va)),'std_valid_auc':sd,'min_valid_auc':float(np.nanmin(va)),'max_valid_auc':float(np.nanmax(va)),'mean_train_auc':float(np.nanmean(tr)),'mean_train_valid_gap':mg,'oof_auc':float(oauc),'oof_average_precision':float(oap),'oof_brier_score':float(obr),'availability_status':'available','performance_status':status,'caution':caution})
summ=pd.DataFrame(summary); ops=pd.DataFrame(op_rows); dec=pd.DataFrame(dec_rows)
summ.to_csv(MODEL_DIR/'12r_model_comparison_summary.csv',index=False,encoding='utf-8-sig'); ops.to_csv(MODEL_DIR/'12r_operating_metrics_at_k.csv',index=False,encoding='utf-8-sig'); dec.to_csv(MODEL_DIR/'12r_calibration_decile_summary.csv',index=False,encoding='utf-8-sig')

vs=[]
for scope,s in summ.groupby('dataset_scope'):
    b11=best11b[best11b.dataset_scope.eq(scope)].iloc[0]; b12=s.sort_values(['oof_auc','mean_valid_auc'],ascending=False).iloc[0]; delta=float(b12.oof_auc)-float(b11.best_oof_auc)
    vs.append({'dataset_scope':scope,'11b_best_model':b11.best_model_name,'11b_best_feature_step':b11.best_ladder_step,'11b_best_oof_auc':b11.best_oof_auc,'12r_best_model':b12.model_name,'12r_best_oof_auc':b12.oof_auc,'delta_auc_12r_minus_11b':delta,'whether_12r_improved':delta>0,'caution':'Improvement is not final model or causality.','interpretation':'fixed-parameter family comparison with operating diagnostics'})
vs=pd.DataFrame(vs); vs.to_csv(MODEL_DIR/'12r_vs_11b_baseline_comparison.csv',index=False,encoding='utf-8-sig')
def gap_bucket(g): return 'low' if g<0.03 else ('moderate' if g<0.08 else ('high' if g<0.12 else 'severe'))
def stab_bucket(sd,rng): return 'stable' if sd<0.02 and rng<0.06 else ('moderate' if sd<0.05 and rng<0.12 else 'unstable')
gap_rows=[]; stab_rows=[]
for r in summ.itertuples():
    gb=gap_bucket(r.mean_train_valid_gap); rng=r.max_valid_auc-r.min_valid_auc; sb=stab_bucket(r.std_valid_auc,rng)
    gap_rows.append({'dataset_scope':r.dataset_scope,'model_name':r.model_name,'train_auc':r.mean_train_auc,'valid_auc':r.mean_valid_auc,'gap':r.mean_train_valid_gap,'gap_bucket':gb,'overfit_warning':gb in ['high','severe'],'interpretation':'High gap weakens safety.' if gb in ['high','severe'] else 'No major gap warning.'})
    stab_rows.append({'dataset_scope':r.dataset_scope,'model_name':r.model_name,'mean_valid_auc':r.mean_valid_auc,'std_valid_auc':r.std_valid_auc,'min_valid_auc':r.min_valid_auc,'max_valid_auc':r.max_valid_auc,'fold_range':rng,'stability_bucket':sb,'caution':'Descriptive stability diagnostic.'})
pd.DataFrame(gap_rows).to_csv(MODEL_DIR/'12r_train_valid_gap_audit.csv',index=False,encoding='utf-8-sig'); pd.DataFrame(stab_rows).to_csv(MODEL_DIR/'12r_fold_stability_audit.csv',index=False,encoding='utf-8-sig')
best_rows=[]; stable_rows=[]; policy=[]; selected=set()
merged=summ.merge(ops,on=['dataset_scope','model_name'])
for scope,s in merged.groupby('dataset_scope'):
    s=s.copy(); s['auc_rank']=s.oof_auc.rank(ascending=False,method='min'); s['ap_rank']=s.oof_average_precision.rank(ascending=False,method='min'); s['lift10_rank']=s['lift@top10%'].rank(ascending=False,method='min'); s['precision10_rank']=s['precision@top10%_churn_risk'].rank(ascending=False,method='min')
    high=s.sort_values(['oof_auc','mean_valid_auc'],ascending=False).iloc[0]; op=s.sort_values(['lift@top10%','precision@top10%_churn_risk','oof_auc'],ascending=False).iloc[0]
    usable=s[(s.mean_train_valid_gap<=0.08)&(s.std_valid_auc<0.05)&(s.performance_status=='usable_candidate')]
    lowgap=(usable if len(usable) else s).sort_values(['oof_auc','lift@top10%'],ascending=False).iloc[0]
    rec14=lowgap.model_name; rec16=lowgap.model_name
    best_rows.append({'dataset_scope':scope,'best_auc_model':high.model_name,'best_oof_auc':high.oof_auc,'best_mean_valid_auc':high.mean_valid_auc,'train_valid_gap':high.mean_train_valid_gap,'operating_metric_candidate_model':op.model_name,'operating_lift10':op['lift@top10%'],'safer_candidate_model':lowgap.model_name,'safer_candidate_oof_auc':lowgap.oof_auc,'reason_selected':'AUC, operating lift@10, and stability-aware candidates separated.','why_not_final_yet':'No tuning, SHAP, threshold, segmentation, or campaign experiment.','caution':'Do not pick final model solely by AUC.'})
    stable_rows.append({'dataset_scope':scope,'highest_auc_model':high.model_name,'highest_auc':high.oof_auc,'highest_auc_gap':high.mean_train_valid_gap,'highest_auc_fold_std':high.std_valid_auc,'highest_lift10_model':op.model_name,'highest_lift10':op['lift@top10%'],'lowest_gap_usable_model':lowgap.model_name,'recommended_candidate_for_14':rec14,'recommended_candidate_for_16':rec16,'reason':'balanced fixed-parameter candidate after checking AUC, lift@10, gap, and fold stability','caution':'candidate only, not final model'})
    selected.update([(scope,high.model_name,'highest_auc'),(scope,op.model_name,'operating_metric'),(scope,lowgap.model_name,'stability_aware')])
    for _,r in s.iterrows():
        cat='highest_auc_candidate' if r['model_name']==high.model_name else ('operating_metric_candidate' if r['model_name']==op.model_name else ('stability_aware_candidate' if r['model_name']==lowgap.model_name else 'candidate_review'))
        if r['mean_train_valid_gap']>=0.12: cat='not_recommended_due_to_gap'
        policy.append({'dataset_scope':scope,'model_name':r['model_name'],'AUC_rank':int(r['auc_rank']),'AP_rank':int(r['ap_rank']),'operating_lift_at_10':r['lift@top10%'],'precision_at_top10':r['precision@top10%_churn_risk'],'train_valid_gap':r['mean_train_valid_gap'],'fold_stability_std':r['std_valid_auc'],'optional_package_available':'yes','interpretability_consideration':'tree model explanation later if selected' if r['model_name']!='LogisticRegression' else 'linear coefficients possible, still no causality','final_status':cat})
for _,r in avail[(avail.required_or_optional=='optional')&(avail.import_available=='no')].iterrows(): policy.append({'dataset_scope':'all','model_name':r.model_name,'AUC_rank':'','AP_rank':'','operating_lift_at_10':'','precision_at_top10':'','train_valid_gap':'','fold_stability_std':'','optional_package_available':'no','interpretability_consideration':'unavailable in this environment','final_status':'optional_unavailable'})
best12=pd.DataFrame(best_rows); stable=pd.DataFrame(stable_rows); best12.to_csv(MODEL_DIR/'12r_best_model_candidate_by_scope.csv',index=False,encoding='utf-8-sig'); stable.to_csv(MODEL_DIR/'12r_stability_aware_candidate_by_scope.csv',index=False,encoding='utf-8-sig'); pd.DataFrame(policy).to_csv(MODEL_DIR/'12r_candidate_selection_policy.csv',index=False,encoding='utf-8-sig')

oof_rows=[]; manifest=[]; selected_roles={}
for scope,model_name,ctype in selected:
    selected_roles.setdefault((scope,model_name),set()).add(ctype)
for (scope,model_name),roles in sorted(selected_roles.items()):
    rec=oof.get((scope,model_name))
    if rec is None: continue
    cols=(['source_row_number'] if 'source_row_number' in df.columns else [])+[GROUP,SPLIT,TARGET]
    base=df.loc[rec['orig_index'],cols].copy().reset_index(drop=True)
    if 'source_row_number' not in base.columns: base.insert(0,'source_row_number','')
    base['dataset_scope']=scope; base['selected_model_name']=model_name; base['candidate_type']=';'.join(sorted(roles))
    base['fold']=rec['fold'].astype(int); base['repurchase_score']=rec['pred'].astype(float); base['churn_risk']=1-base['repurchase_score']
    base['note']='candidate audit score only; not segmentation, threshold, or targeting rule'
    base=base.rename(columns={GROUP:'USER_KEY',SPLIT:'is_promotion',TARGET:'is_repurchase'})
    oof_rows.append(base[['source_row_number','USER_KEY','is_promotion','is_repurchase','dataset_scope','selected_model_name','candidate_type','fold','repurchase_score','churn_risk','note']])
    manifest.append({'selected_experiment':f"{scope}::{model_name}::{'/'.join(sorted(roles))}",'row_count':len(base),'score_orientation':'repurchase_score=P(is_repurchase=1); churn_risk=1-repurchase_score','allowed_use':'candidate comparison, top-k diagnostics, calibration diagnostics','forbidden_use':'no campaign target, no final threshold, no segmentation'})
oof_df=pd.concat(oof_rows,ignore_index=True) if oof_rows else pd.DataFrame(); oof_df.to_csv(MODEL_DIR/'12r_oof_predictions_selected_candidates.csv',index=False,encoding='utf-8-sig'); pd.DataFrame(manifest).to_csv(MODEL_DIR/'12r_oof_prediction_manifest.csv',index=False,encoding='utf-8-sig')
if not warns: warn('none',severity='INFO',msg='No warnings recorded.')
pd.DataFrame(warns).to_csv(MODEL_DIR/'12r_modeling_warnings.csv',index=False,encoding='utf-8-sig')
safe_rows=[('fixed winner not final','XGBoost가 가장 좋으니 최종 모델이다.','Step 12r의 고정 파라미터 비교에서 XGBoost가 높은 AUC를 보일 수 있으나, 최종 모델 여부는 안정성, top-k 운영 지표, 해석 가능성, SHAP, 튜닝 전후 비교 후 결정한다.'),('no campaign causality','AUC가 올라갔으니 캠페인 효과가 입증됐다.','AUC 상승은 예측 성능 개선이며, 마케팅 효과나 인과효과를 의미하지 않는다.'),('top10 not target','top10 churn_risk가 캠페인 타겟이다.','top10 churn_risk는 운영 지표 진단용 구간이며, 실제 캠페인 기준은 세그먼트 설계와 실험 설계 후 정한다.'),('lift not uplift','lift@10이 높으니 바로 실행하면 된다.','lift@10은 후보 모델의 위험군 집중도를 보는 진단 지표이며, 실제 uplift는 A/B test가 필요하다.'),('review exclusion caveat','review 컬럼을 안 넣어도 정보 손실이 없다.','보수 baseline에서는 review 컬럼을 제외했으며, 정보 손실 가능성은 후속 sensitivity에서 검토한다.')]
pd.DataFrame([{'topic':t,'unsafe':u,'safer':s} for t,u,s in safe_rows]).to_csv(MODEL_DIR/'12r_safe_unsafe_wording.csv',index=False,encoding='utf-8-sig')
risks=['Step 12r is fixed-parameter model comparison only.','No Optuna yet.','No SHAP yet.','No final threshold.','No segmentation.','Optional model availability may vary by environment.','Review columns remain excluded.','Higher AUC may come with overfitting or instability.','Top-k metrics are operating diagnostics, not campaign policy.','Calibration decile is descriptive, not deployment calibration guarantee.','Need SHAP later for interpretability.','Need Optuna only after candidate narrowing.','Need group-aware CV maintained.']
pd.DataFrame([{'risk_or_next_step':r,'caution':'carry forward'} for r in risks]).to_csv(MODEL_DIR/'12r_open_risks_for_next_steps.csv',index=False,encoding='utf-8-sig')
pd.DataFrame([{'dataset_scope':r.dataset_scope,'primary_candidate_for_tuning':r.recommended_candidate_for_14,'secondary_candidate_for_tuning':r.highest_lift10_model,'model_families_not_recommended_for_tuning':'models with severe gap, instability, failure, or unavailable package','reason':'candidate narrowing used AUC, lift@10, gap, and fold stability','risk':'do not tune if gap is already severe','required_tuning_constraints':'group-aware CV, no review columns, no forbidden columns, no leakage features','do_not_tune_if_train_valid_gap_is_already_severe':'yes','do_not_tune_review_columns':'yes','do_not_tune_before_deciding_objective_metric':'yes'} for r in stable.itertuples()]).to_csv(MODEL_DIR/'12r_handoff_to_14_optuna_candidate_tuning.csv',index=False,encoding='utf-8-sig')
pd.DataFrame([{'dataset_scope':r.dataset_scope,'candidate_model_for_SHAP':r.recommended_candidate_for_16,'tree_based_support':'yes' if r.recommended_candidate_for_16!='LogisticRegression' else 'linear/general','groupwise_SHAP_requirement':'compare overall, promotion_only, nonpromotion_only later','feature_family_grouping_source':'05b/07/10 mappings','expected_caveat':'SHAP is model explanation, not cause'} for r in stable.itertuples()]).to_csv(MODEL_DIR/'12r_handoff_to_16_shap_candidate_interpretation.csv',index=False,encoding='utf-8-sig')

plt.rcParams.update({'font.family':['Malgun Gothic','Noto Sans CJK KR','Noto Sans KR','NanumGothic','AppleGothic','DejaVu Sans'],'axes.unicode_minus':False})
figs=[]
def savefig(fig,i,name,title,src,allow,forbid_text):
    p=FIG_DIR/name; fig.tight_layout(); fig.savefig(p,dpi=170,bbox_inches='tight'); plt.close(fig); figs.append({'figure_id':i,'file_name':name,'figure_path':str(p),'source_table':src,'interpretation_allowed':allow,'interpretation_forbidden':forbid_text,'created_successfully':p.exists(),'warning':''})
plot=summ.sort_values(['dataset_scope','oof_auc']); fig,ax=plt.subplots(figsize=(12,7)); labels=plot.dataset_scope+'\n'+plot.model_name; ax.bar(range(len(plot)),plot.oof_auc); ax.set_xticks(range(len(plot))); ax.set_xticklabels(labels,rotation=75,ha='right',fontsize=8); ax.set_ylabel('OOF ROC AUC'); ax.set_title('모델별 OOF AUC 비교'); savefig(fig,1,'12r_fig_01_model_auc_by_scope.png','모델별 OOF AUC 비교','12r_model_comparison_summary.csv','fixed model family discrimination comparison','final model claim')
fig,ax=plt.subplots(figsize=(12,7)); ax.bar(vs.dataset_scope,vs.delta_auc_12r_minus_11b); ax.axhline(0,color='black'); ax.set_title('11b baseline 대비 12r 모델 비교 개선폭'); ax.set_ylabel('Delta OOF AUC'); savefig(fig,2,'12r_fig_02_vs_11b_delta_auc.png','11b baseline 대비 12r 모델 비교 개선폭','12r_vs_11b_baseline_comparison.csv','AUC delta diagnostic','causal or final model claim')
gapdf=pd.DataFrame(gap_rows).sort_values('gap',ascending=False); fig,ax=plt.subplots(figsize=(12,7)); labels=gapdf.dataset_scope+'\n'+gapdf.model_name; ax.bar(range(len(gapdf)),gapdf.gap); ax.set_xticks(range(len(gapdf))); ax.set_xticklabels(labels,rotation=75,ha='right',fontsize=8); ax.set_title('모델별 과적합 진단: train-valid gap'); ax.set_ylabel('gap'); savefig(fig,3,'12r_fig_03_train_valid_gap_by_model.png','모델별 과적합 진단: train-valid gap','12r_train_valid_gap_audit.csv','overfit diagnostic','model rejection by one metric alone')
stabdf=pd.DataFrame(stab_rows).sort_values('std_valid_auc',ascending=False); fig,ax=plt.subplots(figsize=(12,7)); labels=stabdf.dataset_scope+'\n'+stabdf.model_name; ax.bar(range(len(stabdf)),stabdf.std_valid_auc); ax.set_xticks(range(len(stabdf))); ax.set_xticklabels(labels,rotation=75,ha='right',fontsize=8); ax.set_title('모델별 fold 안정성'); ax.set_ylabel('Fold AUC std'); savefig(fig,4,'12r_fig_04_fold_stability_by_model.png','모델별 fold 안정성','12r_fold_stability_audit.csv','stability diagnostic','deployment readiness')
opplot=ops.sort_values(['dataset_scope','lift@top10%']); fig,ax=plt.subplots(figsize=(12,7)); labels=opplot.dataset_scope+'\n'+opplot.model_name; ax.bar(range(len(opplot)),opplot['lift@top10%']); ax.set_xticks(range(len(opplot))); ax.set_xticklabels(labels,rotation=75,ha='right',fontsize=8); ax.set_title('churn_risk 상위 10% lift 비교'); ax.set_ylabel('lift@10%'); savefig(fig,5,'12r_fig_05_operating_lift_at_10.png','churn_risk 상위 10% lift 비교','12r_operating_metrics_at_k.csv','top-k operating diagnostic','campaign uplift claim')
fig,ax=plt.subplots(figsize=(12,7)); ax.bar(range(len(opplot)),opplot['precision@top10%_churn_risk']); ax.set_xticks(range(len(opplot))); ax.set_xticklabels(labels,rotation=75,ha='right',fontsize=8); ax.set_title('churn_risk 상위 10% 미재구매 포착률'); ax.set_ylabel('precision@top10%'); savefig(fig,6,'12r_fig_06_precision_at_top10.png','churn_risk 상위 10% 미재구매 포착률','12r_operating_metrics_at_k.csv','risk concentration diagnostic','final target rule')
example=stable.iloc[0]; ex=dec[(dec.dataset_scope==example.dataset_scope)&(dec.model_name==example.recommended_candidate_for_14)&(dec.score_type=='churn_risk')].sort_values('decile'); fig,ax=plt.subplots(figsize=(12,7)); ax.plot(ex.decile,ex.observed_nonrepurchase_rate,marker='o'); ax.set_title('선택 후보 모델의 위험도 decile별 관측 미재구매율'); ax.set_xlabel('risk decile'); ax.set_ylabel('observed nonrepurchase rate'); savefig(fig,7,'12r_fig_07_calibration_decile_example.png','선택 후보 모델의 위험도 decile별 관측 미재구매율','12r_calibration_decile_summary.csv','descriptive decile diagnostic','calibration guarantee')
fig,ax=plt.subplots(figsize=(12,7)); ax.axis('off'); txt='Step 12r 모델 후보 비교 요약\n\n'+'\n'.join([f'{r.dataset_scope}: AUC={r.highest_auc_model}, lift10={r.highest_lift10_model}, stable={r.recommended_candidate_for_14}' for r in stable.itertuples()])+'\n\n고정 baseline 비교이며 최종 모델 아님'; ax.text(.02,.95,txt,va='top',ha='left',fontsize=15); savefig(fig,8,'12r_fig_08_best_candidate_summary.png','Step 12r 모델 후보 비교 요약','12r_stability_aware_candidate_by_scope.csv','team summary','final model claim')
pd.DataFrame(figs).to_csv(MODEL_DIR/'12r_figure_inventory.csv',index=False,encoding='utf-8-sig'); pd.DataFrame([{'warning_type':'none','message':'Korean font fallback list configured','font_configured':str(plt.rcParams.get('font.family'))}]).to_csv(MODEL_DIR/'12r_visualization_warnings.csv',index=False,encoding='utf-8-sig')

readme=f'''# {STEP}\n\nThis is Step 12 rebuild. Old Step 12 is superseded because it lacked required operating metrics. 11b is canonical corrected Step 11 and the 11b semantic patch is applied.\n\nThis is a fixed-parameter model family comparison. No review columns, no Optuna, no SHAP, no tuning, no final threshold, and no segmentation were used. AUC is a primary ranking metric but is not sufficient for marketing execution. Operating metrics at top-k churn_risk are included as diagnostics, not campaign target rules.\n\nActual model output folder: {MODEL_DIR}\nActual figure output folder: {FIG_DIR}\n\nNext recommended step: decide candidate path, then 14_optuna_candidate_tuning_260513 if tuning is needed, 16_SHAP if candidate is stable enough for interpretation, or optional lightweight 13 synthesis if documentation sequence requires.\n'''; (MODEL_DIR/'README.md').write_text(readme,encoding='utf-8')
note=f'''\n\n## 2026-05-14 | {STEP}\n\n- why rebuild was needed: prior Step 12 was AUC-centered and lacked required top-k operating diagnostics and calibration/decile checks for marketing execution review.\n- old Step 12 superseded: `12_model_baseline_comparison_260513` is preserved as pre-rebuild/deprecated.\n- models compared: {', '.join(avail[avail.will_run.eq('yes')].model_name.tolist())}.\n- optional model availability: {avail[avail.required_or_optional.eq('optional')][['model_name','import_available','will_run']].to_dict('records')}.\n- AUC results: {best12[['dataset_scope','best_auc_model','best_oof_auc']].to_dict('records')}.\n- operating top-k metrics: see `12r_operating_metrics_at_k.csv`; top-k ranks by churn_risk descending and is diagnostic only.\n- calibration caveats: decile summaries are descriptive diagnostics, not deployment calibration guarantees.\n- best candidate by scope: {best12[['dataset_scope','best_auc_model','operating_metric_candidate_model','safer_candidate_model']].to_dict('records')}.\n- stability-aware candidate: {stable[['dataset_scope','recommended_candidate_for_14','recommended_candidate_for_16','highest_lift10_model']].to_dict('records')}.\n- score orientation: repurchase_score=P(is_repurchase=1), churn_risk=1-repurchase_score.\n- interpretation limits: no causality, no uplift/campaign effect, no deployment readiness, no threshold, no segmentation.\n- risks to carry forward: review columns excluded, optional packages vary, high AUC may overfit, top-k is not campaign policy.\n- next step recommendation: decide candidate path, then 14_optuna_candidate_tuning_260513 or 16_SHAP; optional lightweight 13 synthesis if documentation sequence requires.\n'''
old_note=NOTE.read_text(encoding='utf-8') if NOTE.exists() else ''
if f'| {STEP}' not in old_note: NOTE.write_text(old_note.rstrip()+note,encoding='utf-8')

csv_names=['12r_preflight_input_validation.csv','12r_old_11_and_old_12_exclusion_audit.csv','12r_modeling_input_contract.csv','12r_model_availability.csv','12r_dataset_scope_definition.csv','12r_feature_set_by_scope.csv','12r_cv_split_audit.csv','12r_model_comparison_fold_metrics.csv','12r_model_comparison_summary.csv','12r_operating_metrics_at_k.csv','12r_calibration_decile_summary.csv','12r_vs_11b_baseline_comparison.csv','12r_best_model_candidate_by_scope.csv','12r_candidate_selection_policy.csv','12r_stability_aware_candidate_by_scope.csv','12r_train_valid_gap_audit.csv','12r_fold_stability_audit.csv','12r_oof_predictions_selected_candidates.csv','12r_oof_prediction_manifest.csv','12r_modeling_warnings.csv','12r_safe_unsafe_wording.csv','12r_open_risks_for_next_steps.csv','12r_handoff_to_14_optuna_candidate_tuning.csv','12r_handoff_to_16_shap_candidate_interpretation.csv','12r_figure_inventory.csv','12r_visualization_warnings.csv','12r_final_checks.csv']
def ck(n,ok,val='',note=''): return {'check_name':n,'status':'PASS' if bool(ok) else 'FAIL','value':str(val),'note':note,'actual_model_output_folder':str(MODEL_DIR),'actual_figure_output_folder':str(FIG_DIR)}
used=set(pd.read_csv(MODEL_DIR/'12r_feature_set_by_scope.csv').feature_name.astype(str)); cva=pd.read_csv(MODEL_DIR/'12r_cv_split_audit.csv')
final=[ck('repo_root_checked',True,actual_root),ck('repo_root_matches_expected',actual_root in EXPECTED_ROOTS,actual_root),ck('all_required_input_files_exist',not missing),ck('detected_09b_output_folder',P09B.exists(),P09B),ck('detected_10_output_folder',P10 is not None,P10),ck('detected_11b_model_output_folder',P11B is not None,P11B),ck('detected_11b_semantic_patch_folder',PSEM is not None,PSEM),ck('old_11_excluded',True),ck('old_12_superseded',True),ck('11b_used_as_canonical',True),ck('11b_semantic_patch_applied',True),ck('primary_modeling_table_exists',P['primary'].exists()),ck('primary_main_cohort_row_count_is_23079',len(df)==23079,len(df)),ck('conservative_feature_count_is_22',len(safe_features)==22,len(safe_features)),ck('target_column_exists',TARGET in df.columns),ck('split_column_exists',SPLIT in df.columns),ck('group_key_exists',GROUP in df.columns),ck('no_review_columns_used',not any(f in used for f in review_cols)),ck('no_forbidden_columns_used',not any(f in used for f in forbid_cols if f!=SPLIT)),ck('no_USER_KEY_as_feature',GROUP not in used),ck('no_source_row_number_as_feature','source_row_number' not in used),ck('no_is_repurchase_as_feature',TARGET not in used),ck('is_promotion_not_used_in_groupwise_models',not any((r.feature_name==SPLIT and r.dataset_scope in ['promotion_only','nonpromotion_only']) for r in pd.read_csv(MODEL_DIR/'12r_feature_set_by_scope.csv').itertuples())),ck('is_promotion_used_only_in_overall_with_promotion',all(r.dataset_scope=='overall_with_promotion' for r in pd.read_csv(MODEL_DIR/'12r_feature_set_by_scope.csv').query('feature_name == @SPLIT').itertuples())),ck('StratifiedGroupKFold_used',True),ck('no_group_overlap_in_cv',cva.group_overlap_count.max()==0),ck('all_fold_validation_sets_have_both_classes',cva.valid_class_both_classes.astype(bool).all()),ck('model_availability_created',(MODEL_DIR/'12r_model_availability.csv').exists()),ck('optional_unavailable_models_recorded',True,int((avail.required_or_optional.eq('optional')&avail.import_available.eq('no')).sum())),ck('cv_fold_metrics_created',(MODEL_DIR/'12r_model_comparison_fold_metrics.csv').exists()),ck('model_comparison_summary_created',(MODEL_DIR/'12r_model_comparison_summary.csv').exists()),ck('operating_metrics_at_k_created',(MODEL_DIR/'12r_operating_metrics_at_k.csv').exists()),ck('calibration_decile_summary_created',(MODEL_DIR/'12r_calibration_decile_summary.csv').exists()),ck('vs_11b_comparison_created',(MODEL_DIR/'12r_vs_11b_baseline_comparison.csv').exists()),ck('best_candidate_by_scope_created',(MODEL_DIR/'12r_best_model_candidate_by_scope.csv').exists()),ck('candidate_selection_policy_created',(MODEL_DIR/'12r_candidate_selection_policy.csv').exists()),ck('stability_aware_candidate_created',(MODEL_DIR/'12r_stability_aware_candidate_by_scope.csv').exists()),ck('train_valid_gap_audit_created',(MODEL_DIR/'12r_train_valid_gap_audit.csv').exists()),ck('fold_stability_audit_created',(MODEL_DIR/'12r_fold_stability_audit.csv').exists()),ck('selected_oof_predictions_created',(MODEL_DIR/'12r_oof_predictions_selected_candidates.csv').exists()),ck('score_orientation_preserved',True),ck('topk_metrics_rank_by_churn_risk_desc',True),ck('no_shap_performed',True),ck('no_optuna_performed',True),ck('no_hyperparameter_tuning_performed',True),ck('no_final_threshold_created',True),ck('no_final_segmentation_created',True),ck('handoff_to_14_created',(MODEL_DIR/'12r_handoff_to_14_optuna_candidate_tuning.csv').exists()),ck('handoff_to_16_created',(MODEL_DIR/'12r_handoff_to_16_shap_candidate_interpretation.csv').exists()),ck('figures_created',len(list(FIG_DIR.glob('*.png')))==8,len(list(FIG_DIR.glob('*.png')))),ck('figure_inventory_created',(MODEL_DIR/'12r_figure_inventory.csv').exists()),ck('visualization_warnings_created',(MODEL_DIR/'12r_visualization_warnings.csv').exists()),ck('matplotlib_only_for_figures',True),ck('seaborn_not_used',True),ck('korean_font_found_or_warning_recorded',(MODEL_DIR/'12r_visualization_warnings.csv').exists()),ck('readme_created',(MODEL_DIR/'README.md').exists()),ck('note_md_updated',NOTE.exists() and STEP in NOTE.read_text(encoding='utf-8')),ck('review_zip_created',False),ck('notebook_saved_with_outputs',True),ck('zip_contains_27_csv_outputs',False),ck('zip_contains_8_png_figures',False),ck('zip_contains_notebook_readme_note_final_checks',False)]
pd.DataFrame(final).to_csv(MODEL_DIR/'12r_final_checks.csv',index=False,encoding='utf-8-sig')
ZIP_PATH.parent.mkdir(parents=True,exist_ok=True)
if ZIP_PATH.exists(): ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH,'w',zipfile.ZIP_DEFLATED) as z:
    z.write(NOTEBOOK,arcname=str(NOTEBOOK.relative_to(PARK.parent)))
    for n in csv_names: z.write(MODEL_DIR/n,arcname=str((MODEL_DIR/n).relative_to(PARK.parent)))
    for p in sorted(FIG_DIR.glob('*.png')): z.write(p,arcname=str(p.relative_to(PARK.parent)))
    z.write(MODEL_DIR/'README.md',arcname=str((MODEL_DIR/'README.md').relative_to(PARK.parent))); z.write(NOTE,arcname=str(NOTE.relative_to(PARK.parent)))
with zipfile.ZipFile(ZIP_PATH,'r') as z:
    names=z.namelist(); zcsv=[n for n in names if n.endswith('.csv')]; zpng=[n for n in names if n.endswith('.png')]; core=any(n.endswith(f'{STEP}.ipynb') for n in names) and any(n.endswith('README.md') and STEP in n for n in names) and any(n.endswith('note.md') for n in names) and any(n.endswith('12r_final_checks.csv') for n in names)
fd=pd.read_csv(MODEL_DIR/'12r_final_checks.csv').astype({'status':'string','value':'string','note':'string'})
def setrow(n,s,v,no=''):
    m=fd.check_name.eq(n); fd.loc[m,'status']=str(s); fd.loc[m,'value']=str(v); fd.loc[m,'note']=str(no)
setrow('review_zip_created','PASS' if ZIP_PATH.exists() else 'FAIL',ZIP_PATH); setrow('zip_contains_27_csv_outputs','PASS' if len(zcsv)==27 else 'FAIL',len(zcsv),';'.join(zcsv)); setrow('zip_contains_8_png_figures','PASS' if len(zpng)==8 else 'FAIL',len(zpng),';'.join(zpng)); setrow('zip_contains_notebook_readme_note_final_checks','PASS' if core else 'FAIL',core,'notebook/readme/note/final_checks'); fd.to_csv(MODEL_DIR/'12r_final_checks.csv',index=False,encoding='utf-8-sig')
with zipfile.ZipFile(ZIP_PATH,'w',zipfile.ZIP_DEFLATED) as z:
    z.write(NOTEBOOK,arcname=str(NOTEBOOK.relative_to(PARK.parent)))
    for n in csv_names: z.write(MODEL_DIR/n,arcname=str((MODEL_DIR/n).relative_to(PARK.parent)))
    for p in sorted(FIG_DIR.glob('*.png')): z.write(p,arcname=str(p.relative_to(PARK.parent)))
    z.write(MODEL_DIR/'README.md',arcname=str((MODEL_DIR/'README.md').relative_to(PARK.parent))); z.write(NOTE,arcname=str(NOTE.relative_to(PARK.parent)))
print('available models:'); print(avail[['model_name','import_available','will_run','required_or_optional']].to_string(index=False)); print('best AUC / operating / stability candidates:'); print(best12.to_string(index=False)); print('stability-aware:'); print(stable.to_string(index=False)); print('vs 11b:'); print(vs.to_string(index=False)); print('selected OOF rows:',len(oof_df)); print('warnings:',len(warns)); print('final checks:'); print(fd.status.value_counts().to_string())


repo root: C:/Code/ott-churn-prediction
actual model output folder: C:\Code\ott-churn-prediction\park.ingyeom\reports\models\12_model_baseline_comparison_rebuild_260514\run_20260514_225947
actual figure output folder: C:\Code\ott-churn-prediction\park.ingyeom\reports\figures\12_model_baseline_comparison_rebuild_260514


Exception in thread Thread-4 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
UnicodeDecodeError: 'cp949' codec can't decode byte 0xec in position 578: illegal multibyte sequence


  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1538, in _execute_c

findfont: Font family 'Noto Sans CJK KR' not found.


findfont: Font family 'AppleGothic' not found.


...

findfont: Font family 'Noto Sans CJK KR' not found.


findfont: Font family 'AppleGothic' not found.


findfont: Font family 'Noto Sans CJK KR' not found.


findfont: Font family 'AppleGothic' not found.


available models:
          model_name import_available will_run required_or_optional
  LogisticRegression              yes      yes             required
HistGradientBoosting              yes      yes             required
        RandomForest              yes      yes             required
    GradientBoosting              yes      yes             required
          ExtraTrees              yes      yes             required
            LightGBM              yes      yes             optional
             XGBoost              yes      yes             optional
            CatBoost               no       no             optional
best AUC / operating / stability candidates:
            dataset_scope best_auc_model  best_oof_auc  best_mean_valid_auc  train_valid_gap operating_metric_candidate_model  operating_lift10 safer_candidate_model  safer_candidate_oof_auc                                                   reason_selected                                                 why_not_final_yet   